In [1]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: CUPY_ACCELERATORS=cutensor,cub


In [2]:
import tensorly as tl
import plotly.io as pio
#pio.renderers.default = 'iframe'
tl.set_backend('numpy')
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')

TensorLy backend: numpy


In [3]:
from moabb.datasets import *
from moabb.paradigms import P300

dataset = BNCI2014_008()
paradigm = P300(tmin=-0.2)
X, y, meta = paradigm.get_data(dataset)
groups=meta['subject']
X = tl.tensor(X)
X.shape

/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 4200 events (all good), -0.199 – 1 s (baseline off), ~79.0 MiB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")
/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 4200 events (all good), -0.199 – 1 s (baseline off), ~79.0 MiB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")
/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropr

(33600, 8, 308)

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from hoda.classification import SelectFCutoff
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

clf = make_pipeline(
    FunctionTransformer(tl.to_numpy),
    PCA(n_components=None, whiten=True),
    SelectFCutoff(cutoff=1),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

In [5]:
hoda_params=  dict(
    max_iter=64,
    toeplitz=(1,),
    taper=False,
    verbose=True,
    refit_shrinkage=True,
)

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(n_splits=5)

bttdacv_params = dict(
    hoda_params=hoda_params,
    verbose=True,
    cv=cv,
    n_jobs=1,
    clf = clf,
)

In [7]:
from hoda.classification import BTTDACV

bttdacv = BTTDACV(
    max_n_blocks=1,
    fixed_n_blocks=True,
    thetas=[0,0.1,0.2,0.3,0.4,0.5,0.6, 0.7, 0.8, 0.9,1],
    **bttdacv_params
)

In [ ]:
bttdacv.fit(X,y, groups=groups)

fold=0, theta=0
Fitting block 1/1...


Forward model :   8%|████▏                                               | 5/63 [00:00<00:06,  9.30it/s]


fold=0, theta=0, n_blocks=1
fold=0, theta=0.1
Fitting block 1/1...


Forward model :  11%|█████▊                                              | 7/63 [00:00<00:04, 11.66it/s]


fold=0, theta=0.1, n_blocks=1
fold=0, theta=0.2
Fitting block 1/1...


...

Forward model :  21%|██████████▌                                        | 13/63 [00:02<00:09,  5.16it/s]


fold=0, theta=0.6, n_blocks=1
fold=0, theta=0.7
Fitting block 1/1...


Forward model :  22%|███████████▎                                       | 14/63 [00:02<00:09,  5.10it/s]


fold=0, theta=0.7, n_blocks=1
fold=0, theta=0.8
Fitting block 1/1...


Forward model :  49%|█████████████████████████                          | 31/63 [00:15<00:16,  1.97it/s]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

for b in bttdacv.blocks_:
    plt.plot(tl.to_numpy(b.aps_[1]))
    plt.show()